In [3]:
import findspark
findspark.init()

from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

conf = SparkConf().setAppName('1321').setMaster('local[4]')
spark = SparkSession.builder.config(conf = conf).getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/08 01:16:41 WARN Utils: Your hostname, de24, resolves to a loopback address: 127.0.1.1; using 192.168.0.103 instead (on interface enp0s3)
25/08/08 01:16:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/08 01:16:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
'''
Table: Customer

+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| customer_id   | int     |
| name          | varchar |
| visited_on    | date    |
| amount        | int     |
+---------------+---------+
In SQL,(customer_id, visited_on) is the primary key for this table.
This table contains data about customer transactions in a restaurant.
visited_on is the date on which the customer with ID (customer_id) has visited the restaurant.
amount is the total paid by a customer.
 

You are the restaurant owner and you want to analyze a possible expansion 
(there will be at least one customer every day).

Compute the moving average of how much the customer paid in a seven days window 
(i.e., current day + 6 days before). average_amount should be rounded to two decimal places.

Return the result table ordered by visited_on in ascending order.

The result format is in the following example.
Example 1:

Input: 
Customer table:
+-------------+--------------+--------------+-------------+
| customer_id | name         | visited_on   | amount      |
+-------------+--------------+--------------+-------------+
| 1           | Jhon         | 2019-01-01   | 100         |
| 2           | Daniel       | 2019-01-02   | 110         |
| 3           | Jade         | 2019-01-03   | 120         |
| 4           | Khaled       | 2019-01-04   | 130         |
| 5           | Winston      | 2019-01-05   | 110         | 
| 6           | Elvis        | 2019-01-06   | 140         | 
| 7           | Anna         | 2019-01-07   | 150         |
| 8           | Maria        | 2019-01-08   | 80          |
| 9           | Jaze         | 2019-01-09   | 110         | 
| 1           | Jhon         | 2019-01-10   | 130         | 
| 3           | Jade         | 2019-01-10   | 150         | 
+-------------+--------------+--------------+-------------+
Output: 
+--------------+--------------+----------------+
| visited_on   | amount       | average_amount |
+--------------+--------------+----------------+
| 2019-01-07   | 860          | 122.86         |
| 2019-01-08   | 840          | 120            |
| 2019-01-09   | 840          | 120            |
| 2019-01-10   | 1000         | 142.86         |
+--------------+--------------+----------------+
Explanation: 
1st moving average from 2019-01-01 to 2019-01-07 has an average_amount of (100 + 110 + 120 + 130 + 110 + 140 + 150)/7 = 122.86
2nd moving average from 2019-01-02 to 2019-01-08 has an average_amount of (110 + 120 + 130 + 110 + 140 + 150 + 80)/7 = 120
3rd moving average from 2019-01-03 to 2019-01-09 has an average_amount of (120 + 130 + 110 + 140 + 150 + 80 + 110)/7 = 120
4th moving average from 2019-01-04 to 2019-01-10 has an average_amount of (130 + 110 + 140 + 150 + 80 + 110 + 130 + 150)/7 = 142.86
'''

In [4]:
data = [
(1,'Jhon'    ,'2019-01-01',100),
(2,'Daniel'  ,'2019-01-02',110),
(3,'Jade'    ,'2019-01-03',120),
(4,'Khaled'  ,'2019-01-04',130),
(5,'Winston' ,'2019-01-05',110), 
(6,'Elvis'   ,'2019-01-06',140), 
(7,'Anna'    ,'2019-01-07',150),
(8,'Maria'   ,'2019-01-08',80 ),
(9,'Jaze'    ,'2019-01-09',110), 
(1,'Jhon'    ,'2019-01-10',130), 
(3,'Jade'    ,'2019-01-10',150)
]
schema = ['customer_id','name','visited_on','amount']

In [5]:
df = spark.createDataFrame(data= data, schema = schema)
df.show()

+-----------+-------+----------+------+
|customer_id|   name|visited_on|amount|
+-----------+-------+----------+------+
|          1|   Jhon|2019-01-01|   100|
|          2| Daniel|2019-01-02|   110|
|          3|   Jade|2019-01-03|   120|
|          4| Khaled|2019-01-04|   130|
|          5|Winston|2019-01-05|   110|
|          6|  Elvis|2019-01-06|   140|
|          7|   Anna|2019-01-07|   150|
|          8|  Maria|2019-01-08|    80|
|          9|   Jaze|2019-01-09|   110|
|          1|   Jhon|2019-01-10|   130|
|          3|   Jade|2019-01-10|   150|
+-----------+-------+----------+------+



In [15]:
from pyspark.sql.window import Window

windows = Window.orderBy(F.col("visited_on")).rowsBetween(-6,0)

temp_df = df.groupBy(F.col("visited_on"))\
            .agg(F.sum(F.col("amount")).alias("amount"))
# temp_df.show()
temp1_df = temp_df.select(
                            F.col("visited_on"),
                            F.sum(F.col("amount")).over(windows).alias("amount"),
                            F.round(F.avg(F.col("amount")).over(windows),2).alias("average_amount")
                          )
min_date = df.select(F.min(F.col("visited_on"))).collect()[0][0]

# temp1_df.show()
temp1_df.where(
        F.datediff( F.col("visited_on"), 
                     F.lit(min_date)) >= 6
                    )\
        .show()

25/08/08 02:09:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/08 02:09:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/08 02:09:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/08 02:09:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/08 02:09:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/08 02:09:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/08 0

+----------+------+--------------+
|visited_on|amount|average_amount|
+----------+------+--------------+
|2019-01-07|   860|        122.86|
|2019-01-08|   840|         120.0|
|2019-01-09|   840|         120.0|
|2019-01-10|  1000|        142.86|
+----------+------+--------------+



## SQL Solution

<pre>
WITH TEMP
AS (
	SELECT visited_on
		,SUM(amount) AS amount
	FROM Customer
	GROUP BY visited_on
	)
	,TEMP_1
AS (
	SELECT visited_on
		,sum(amount) OVER (
			ORDER BY visited_on rows 6 preceding 
			) amount
		,ROUND(AVG(amount) OVER (
				ORDER BY visited_on rows 6 preceding
				), 2) AS average_amount
	FROM TEMP
	)
SELECT visited_on
	,amount
	,average_amount
FROM TEMP_1
WHERE DATEDIFF(visited_on, (
			SELECT MIN(visited_on)
			FROM Customer
			)) >= 6
ORDER BY visited_on;
</pre>